# Diagnóstico financeiro da Olist: evolução do faturamento e saúde do crescimento

In [1]:
import queries_sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

In [2]:
def format_brl(value: float) -> str:
    """
    Formata valores monetários no padrão brasileiro para exibição nos indicadores.

    Args:
        value (float): Valor numérico a ser formatado.

    Returns:
        str: String formatada no padrão monetário BRL (ex: R$ 1.234,56).
    """
    # Executa a formatação padrão e ajusta os separadores para o padrão brasileiro.
    return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def format_number(value: np.int64 | float) -> str:
    """
    Formata números inteiros ou de ponto flutuante para o padrão brasileiro de exibição.

    Args:
        value (np.int64 | float): Valor numérico a ser formatado.

    Returns:
        str: String formatada com separadores de milhar e decimal adaptados.
    """
    if float(value).is_integer():
        # Formata valores inteiros aplicando o separador de milhar.
        return f"{value:,.0f}".replace(",", ".")
    else:
        # Arredonda e formata valores decimais aplicando os separadores brasileiros.
        return f"{np.round(value, 2):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def calc_delta(metric_2017: int | float, metric_2018: int | float) -> tuple:
    """
    Calcula a diferença absoluta e percentual entre os períodos comparados.

    Args:
        metric_2017 (int | float): Valor de referência do ano de 2017.
        metric_2018 (int | float): Valor de comparação do ano de 2018.

    Returns:
        tuple: Tupla contendo a diferença absoluta e a variação percentual.
    """
    delta = metric_2018 - metric_2017
    delta_percentage = delta / metric_2017 * 100
    return delta, delta_percentage

def kpi_card(title: str, value: str, description: str = None) -> None:
    """
    Exibe indicadores formatados em cartões HTML customizados no ambiente de notebook.

    Args:
        title (str): Título ou nome do indicador.
        value (str): Valor principal a ser destacado no cartão.
        description (str, optional): Texto descritivo opcional (ex: indicativo de variação).
    """
    if description == None:
        # Constrói o layout HTML padrão para cartões sem descrição ou variação detalhada.
        html_code = f"""
                <div style="
                    background-color: #f8f9fa; 
                    border-left: 5px solid #2b5d8d; 
                    padding: 15px 20px; 
                    border-radius: 6px; 
                    margin: 10px 0; 
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.05);
                ">
                    <span style="font-size: 15px; color: #6c757d; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">{title}</span>
                    <div style="font-size: 34px; font-weight: 700; color: #2c3e50; margin: 5px 0;">{value}</div>
                </div>
                """
    else:
        if "↓" in description:
            color = "#dc3545" # Define a cor vermelha para indicar queda.
        else:
            color = "#28a745" # Define a cor verde para indicar aumento, estabilidade ou outros casos.

        # Constrói o layout HTML completo incluindo o rodapé com a descrição e coloração condicional.
        html_code = f"""
            <div style="
                background-color: #f8f9fa; 
                border-left: 5px solid #2b5d8d; 
                padding: 15px 20px; 
                border-radius: 6px; 
                margin: 10px 0; 
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                box-shadow: 0 2px 4px rgba(0,0,0,0.05);
            ">
                <span style="font-size: 15px; color: #6c757d; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">{title}</span>
                <div style="font-size: 34px; font-weight: 700; color: #2c3e50; margin: 5px 0;">{value}</div>
                {'<div style="font-size: 12px; color:' + color + '; font-weight: 500;">' + description + '</div>'}
            </div>
            """
    display(HTML(html_code))

# Análise Exploratória: Como o faturamento evoluiu?

In [3]:
# Carrega os dados
# Consulta os pagamentos agregados por mês para acompanhar a evolução da receita.
sales_data = sql.get_sales_data(step="%Y-%m")

sales_data["date"] = pd.to_datetime(sales_data["date"])

In [4]:
sales_data.head()

,date,revenue,sales_count
0,2016-12-01,19.62,1
1,2017-01-01,121947.60,715
2,2017-02-01,270809.17,1638
3,2017-03-01,410773.10,2554
4,2017-04-01,387909.63,2278


In [5]:
revenue = sales_data["revenue"].sum()

kpi_card(
    title = 'Faturamento Total', 
    value = format_brl(revenue)
)

In [6]:
fig = px.line(
    sales_data,
    x="date",
    y="revenue",
    title="Evolução do Faturamento — Olist",
    labels={
        "period": "Data",
        "revenue": "Faturamento (R$)"
    },
    template="plotly_white",
    color_discrete_sequence=["#2563EB"],
)

fig.update_layout(
    title=dict(
        x=0.5,  # Centraliza o título
        xanchor="center"
    ),
    xaxis_title="Data",
    yaxis_title="Faturamento (R$)",
    hovermode="x unified",
    margin=dict(l=60, r=30, t=80, b=60),
    separators = ",."
)

fig.update_xaxes(
    showgrid=False,
    tickformat="%b %Y"
)

fig.update_yaxes(
    tickprefix="R$ ",
    tickformat=",.2f",
    separatethousands=True,
    gridcolor="lightgray"
)

fig.show()

In [7]:
# Filtra períodos equivalentes de 2017 e 2018 para comparar as métricas.
filter_2017 = (
    (sales_data["date"] >= "2017-01-01")
    & (sales_data["date"] <= "2017-08-26")
)

filter_2018 = (
    (sales_data["date"] >= "2018-01-01")
    & (sales_data["date"] <= "2018-08-26")
)

sales_data_2017 = sales_data[filter_2017].copy()
sales_data_2018 = sales_data[filter_2018].copy()


In [8]:
revenue_2017 = sales_data_2017["revenue"].sum()
revenue_2018 = sales_data_2018["revenue"].sum()

delta_revenue, delta_revenue_percentage = calc_delta(revenue_2017, revenue_2018)

In [9]:
kpi_card(
    title = 'Faturamento em 2017', 
    value = format_brl(revenue_2017)
)

In [10]:
kpi_card(
    title = 'Faturamento em 2018', 
    value = format_brl(revenue_2018),
    description = f'+{format_brl(delta_revenue)} ↑ {delta_revenue_percentage:.2f}% em relação ao mesmo periodo anterior'
)

# O crescimento da receita é saudável?

> **Observamos um crescimento da receita na análise anterior: o faturamento aumentou 143,87% em relação ao mesmo período do ano anterior. Esse resultado levanta a seguinte questão: o crescimento da receita é saudável?"**

Em nosso contexto, definiremos crescimento saudável como aquele em que a receita é recorrente e não está concentrada em poucos clientes. Com essa definição, podemos avaliar a "saúde" do crescimento respondendo a perguntas mais simples:
- Qual é o mecanismo de crescimento da receita? O aumento das vendas ocorre pela conquista de novos clientes, pelo aumento da frequência de recompra ou pelo aumento do valor das vendas?
- A receita está concentrada?

## Qual o mecanismo de crescimento da receita?

> **O crescimento da receita pode ocorrer por diferentes mecanismos: pelo aumento do valor médio gasto em cada compra ou pelo aumento do volume de vendas realizadas. Para identificar qual desses fatores contribuiu para o crescimento observado, utilizaremos as seguintes métricas:**

- **Ticket médio:** permite avaliar se os clientes passaram a gastar mais em cada compra, indicando um aumento no valor médio das vendas.
- **Quantidade de vendas:** permite verificar se houve um aumento no volume de transações, indicando maior atividade comercial.

A análise conjunta dessas métricas permite compreender se o crescimento do faturamento foi impulsionado principalmente pelo aumento do valor médio das compras, pelo maior volume de vendas ou pela combinação dos dois fatores.

In [11]:
ticket_value_per_period = sales_data["revenue"] / sales_data["sales_count"]

In [12]:
fig = px.line(
    x=sales_data["date"],
    y=ticket_value_per_period,
    title="Evolução do Ticket Médio — Olist",
    labels={
        "period": "Data",
        "revenue": "Ticket Médio (R$)"
    },
    template="plotly_white",
    color_discrete_sequence=["#2563EB"],
)

fig.update_layout(
    title=dict(
        x=0.5,  # Centraliza o título
        xanchor="center"
    ),
    xaxis_title="Data",
    yaxis_title="Ticket Médio (R$)",
    hovermode="x unified",
    margin=dict(l=60, r=30, t=80, b=60),
    separators = ",."
)

fig.update_xaxes(
    showgrid=False,
    tickformat="%b %Y"
)

fig.update_yaxes(
    tickprefix="R$ ",
    tickformat=",.2f",
    separatethousands=True,
    gridcolor="lightgray"
)

fig.show()

O ticket se estabiliza ao longo do tempo, tornando a média uma medida representativa do comportamento das vendas.

In [13]:
number_of_sales = sales_data["sales_count"].sum()
average_ticket_value = revenue / number_of_sales

In [14]:
kpi_card(
    title = "Ticket Médio",
    value = format_brl(average_ticket_value)
)

In [15]:
kpi_card(
    title = "Quantidade total de vendas",
    value = format_number(number_of_sales)
)

In [16]:
number_of_sales_2017 = sales_data_2017["sales_count"].sum()
number_of_sales_2018 = sales_data_2018["sales_count"].sum()

delta_sales, delta_sales_percentage = calc_delta(number_of_sales_2017, number_of_sales_2018)

In [17]:
average_ticket_value_2017 = revenue_2017 / number_of_sales_2017
average_ticket_value_2018 = revenue_2018 / number_of_sales_2018

delta_average_ticket_value, delta_average_ticket_value_percentage = calc_delta(average_ticket_value_2017, average_ticket_value_2018)

In [18]:
kpi_card(
    title = 'Quantidade de vendas em 2017', 
    value = format_number(number_of_sales_2017)
)

In [19]:
kpi_card(
    title = 'Quantidade de vendas em 2018', 
    value = format_number(number_of_sales_2018),
    description = f'+{format_number(delta_sales)} ↑ {delta_sales_percentage:.2f}% em relação ao mesmo periodo anterior'
)

In [20]:
kpi_card(
    title = "Ticket médio em 2017",
    value = format_brl(average_ticket_value_2017)
)

In [21]:
kpi_card(
    title = "Ticket médio em 2018",
    value = format_brl(average_ticket_value_2018),
    description = f'+{format_brl(delta_average_ticket_value)} ↑ {delta_average_ticket_value_percentage:.2f}% em relação ao mesmo periodo anterior'
)

**Resposta:** A receita aumentou devido ao crescimento do número de vendas. O ticket médio permaneceu praticamente estável (+1,53%), enquanto o volume de vendas cresceu 140,19% no mesmo período.

## A base de clientes cresceu ou a frequência de compra aumentou?

> **Observamos anteriormente que o volume de vendas aumentou. Esse crescimento, porém, pode ter ocorrido por dois motivos: a conquista de novos clientes ou o aumento da frequência de compra dos clientes já existentes. Para identificar qual desses fatores contribuiu para o resultado, utilizaremos as seguintes métricas:**

- **Quantidade de clientes:** permite avaliar a evolução da base de clientes, indicando se o crescimento do volume de vendas está associado à conquista de novos consumidores.
- **Frequência de compra dos clientes:** permite identificar quantas vezes, em média, cada cliente realizou compras em determinado período, indicando se os clientes existentes passaram a comprar com maior frequência.

A análise conjunta dessas métricas permite compreender se o aumento do volume de vendas foi impulsionado principalmente pela expansão da base de clientes, pelo aumento da frequência de compra ou pela combinação dos dois fatores.

In [22]:
# Obtém as datas e os identificadores de clientes para analisar a base e a frequência de compra.
customer_data = sql.get_clients_data() 

In [23]:
customer_data.head()

,date,customer_id
0,2016-12-23 23:31:37,f7b981e8a280e455ac3cbe0d5d171bd1
1,2017-01-05 12:10:17,f7b981e8a280e455ac3cbe0d5d171bd1
2,2017-01-05 16:15:16,4b3207464f5f7a48a7f63fa0b1251d86
3,2017-01-05 23:05:27,b88b78a413e70182e18b032361b24f91
4,2017-01-06 13:55:11,48368b31672665cca1b8a03047a1702a


In [24]:
customer_data["new_customer"] = (
    ~customer_data["customer_id"].duplicated() # Identifica clientes que já apareceram anteriormente e ~ inverte o resultado, identificando a primeira ocorrência de cada cliente.
)

customer_data["new_customer"].astype(int) # Transforma True em 1 e False em 0

base_growth = customer_data["new_customer"].cumsum()
revenue_growth = sales_data["revenue"].cumsum()

In [25]:
fig = go.Figure()

# Eixo Y esquerdo: quantidade de clientes
fig.add_trace(go.Scatter(
    x=customer_data["date"],
    y=base_growth,
    mode="lines",
    name="Base de clientes",
    line=dict(color="#2563EB"),
    yaxis="y"
))

# Eixo Y direito: faturamento em reais
fig.add_trace(go.Scatter(
    x=sales_data["date"],
    y=revenue_growth,
    mode="lines",
    name="Faturamento (R$)",
    line=dict(color="#F59E0B"),
    yaxis="y2"
))

fig.update_layout(
    title=dict(
        text="Evolução de Clientes e Faturamento — Olist",
        x=0.5,
        xanchor="center"
    ),
    template="plotly_white",
    hovermode="x unified",

    # Eixo Y esquerdo
    yaxis=dict(
        title="Base de clientes",
        tickformat=",.0f",
        separatethousands=True,
        gridcolor="lightgray",
        rangemode="tozero"
    ),

    # Eixo Y direito
    yaxis2=dict(
        title="Faturamento Acumulado (R$)",
        tickprefix="R$ ",
        tickformat=",.2f",
        separatethousands=True,
        overlaying="y",
        side="right",
        showgrid=False,
        rangemode="tozero"
    ),

    xaxis=dict(
        title="Data",
        showgrid=False,
        tickformat="%b %Y"
    ),

    margin=dict(l=80, r=100, t=80, b=60),
    separators=",."
)

fig.show()

O faturamento cresce praticamente no mesmo ritmo que a base de clientes.

In [26]:
purchases_per_customer = (
    customer_data.groupby("customer_id")
    .size()
)

average_freq = purchases_per_customer.mean()

In [27]:
fig = px.histogram(
    x=purchases_per_customer,
    nbins=8,
    title="Distribuição da Frequência de Compras por Cliente",
    labels={
        "x": "Número de compras por cliente",
        "y": "Quantidade de clientes"
    },
    template="plotly_white",
    color_discrete_sequence=["#2563EB"]
)

fig.update_layout(
    title=dict(x=0.5, xanchor="center"),
    xaxis_title="Número de compras por cliente",
    yaxis_title="Quantidade de clientes",
    bargap=0.1
)

fig.update_traces(
    texttemplate="%{y}",       
    textposition="outside",    
    cliponaxis=False,          
    selector=dict(type="histogram") 
)

fig.show()

In [28]:
kpi_card(
    title = "Frequência média de compras por cliente",
    value = format_number(average_freq)
)

In [29]:
customer_data["date"] = pd.to_datetime(customer_data["date"])

filter_2017 = (
    (customer_data["date"] >= "2017-01-01")
    & (customer_data["date"] <= "2017-08-26")
    )

filter_2018 = (
    (customer_data["date"] >= "2018-01-01")
    & (customer_data["date"] <= "2018-08-26")
)

customer_data_2018 = customer_data[filter_2018].copy()
customer_data_2017 = customer_data[filter_2017].copy()


purchases_per_customer_2017 = (
    customer_data_2017.groupby("customer_id")
    .size()
)

purchases_per_customer_2018 = (
    customer_data_2018.groupby("customer_id")
    .size()
)

average_freq_2017 = purchases_per_customer_2017.mean()
average_freq_2018 = purchases_per_customer_2018.mean()

delta_average_freq, delta_freq_percentage = calc_delta(average_freq_2017, average_freq_2018)

In [30]:
kpi_card(
    title = "Frequência média de compras por cliente em 2017",
    value = format_number(average_freq_2017)
)

In [31]:
kpi_card(
    title = "Frequência média de compras por cliente em 2018",
    value = format_number(average_freq_2018),
    description = f'{delta_average_freq:.3f} ↓ {delta_freq_percentage:.2f}% em relação ao mesmo periodo anterior'
)

In [32]:
total_customers = customer_data["new_customer"].sum()
total_customers_2017 = customer_data_2017["new_customer"].sum()
total_customers_2018 = customer_data_2018["new_customer"].sum()

delta_customers, delta_customers_percentage = calc_delta(total_customers_2017, total_customers_2018)

In [33]:
kpi_card(
    title = "Quantidade de clientes",
    value = format_number(total_customers)
)

In [34]:
kpi_card(
    title = "Quantidade de clientes em 2017",
    value = format_number(total_customers_2017)
)

In [35]:
kpi_card(
    title = "Quantidade de clientes em 2018",
    value = format_number(total_customers_2018),
    description = f'+{format_number(delta_customers)} ↑ {delta_customers_percentage:.2f}% em relação ao mesmo periodo anterior'
)

**Resposta:** O número de vendas aumentou em decorrência do crescimento de 147,17% na base de clientes, e não do aumento da frequência de compra, que permaneceu praticamente estável, apresentando uma variação negativa de 0,45%.

**Recomendação:**

Criar programas de fidelidade, desenvolver campanhas de recompra, identificar clientes inativos e elaborar campanhas personalizadas com base no histórico de compras, buscando aumentar a frequência de compra dos clientes atuais. A empresa conseguiu conquistar novos clientes, mas ainda não demonstrou crescimento significativo na recorrência de compra.

## Poucos clientes concentram muita receita?

> **O crescimento da receita pode parecer positivo, mas é importante avaliar se o faturamento está distribuído entre uma parcela significativa da base de clientes ou se depende de poucos consumidores. Uma elevada concentração pode representar um risco para a sustentabilidade da receita, caso alguns clientes deixem de comprar. Para investigar essa questão, utilizaremos as seguintes métricas:**

- **Faturamento por cliente:** permite identificar quanto cada cliente contribui para a receita total, evidenciando diferenças na participação individual dos consumidores no faturamento.
- **Concentração da receita por número de clientes:** permite avaliar qual parcela do faturamento está concentrada em diferentes proporções da base de clientes, identificando o grau de dependência em relação aos maiores consumidores.

A análise conjunta dessas métricas permite compreender como a receita está distribuída entre os clientes, identificar possíveis concentrações e avaliar o nível de dependência do negócio em relação a um grupo restrito de consumidores.

In [36]:
# Obtém o faturamento agregado por cliente para avaliar a concentração da receita.
sales_by_client_data = sql.get_sales_by_clients_data()

In [37]:
sales_by_client_data.head()

,customer_id,revenue
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


In [38]:
sales_by_client_data.describe()

,revenue
count,92910.000000
mean,165.242072
std,226.519415
min,9.590000
25%,63.100000
50%,107.820000
75%,182.600000
max,13664.080000


In [39]:
# Calcula a quantidade de clientes correspondente a 75% da base
total_clients = len(sales_by_client_data)
clients_75_percent = int(total_clients * 0.75)

# Calcula o faturamento correspondente ao terceiro quartil (Q3)
revenue_q3 = sales_by_client_data["revenue"].quantile(0.75)

# Exibe o resultado
print(
    f"75% dos clientes ({format_number(clients_75_percent)}) "
    f"possuem faturamento igual ou inferior a {format_brl(revenue_q3)}."
)

75% dos clientes (69.682) possuem faturamento igual ou inferior a R$ 182,60.


In [40]:
sales_by_client_data.describe(include="object") 

,customer_id
count,92910
unique,92910
top,0000366f3b9a7992bf8c76cfdf3221e2
freq,1


In [41]:
# Calcula a média do faturamento
mean_revenue = sales_by_client_data["revenue"].mean()

In [42]:
kpi_card(
    title="Faturamento médio por cliente",
    value=format_brl(mean_revenue)
)

In [43]:
# Calcula o faturamento do cliente que mais comprou
max_revenue = sales_by_client_data["revenue"].max()

In [44]:
kpi_card(
    title="Faturamento máximo gerado por um único cliente.",
    value=format_brl(max_revenue)
)

In [45]:
# 1. Cria o Histograma com o Box Plot na margem
fig = px.histogram(
    data_frame=sales_by_client_data,
    x="revenue",
    nbins=18,
    marginal="box", 
    title="Distribuição do Faturamento por Cliente",
    template="plotly_white",
    color_discrete_sequence=["#2563EB"]
)

# 2A. Adiciona a linha vertical
fig.add_vline(
    x=mean_revenue,
    line_dash="dash",
    line_color="#EF4444",  
    line_width=2,
)

# 2B. Adiciona o nome/rótulo da linha na legenda (já que shapes não entram na legenda sozinhos)
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode="lines",
    name=f"Média: {format_brl(mean_revenue)}",  # O "nome" que aparece na legenda
    line=dict(color="#EF4444", dash="dash", width=2), # Mesmo estilo da linha vermelha tracejada
    showlegend=True
))

# 3. Força a exibição dos números (incluindo o zero) em cima de cada barra
fig.update_traces(
    texttemplate="%{y}",       
    textposition="outside",    
    cliponaxis=False,          
    selector=dict(type="histogram") 
)

# 4. Ajustes de layout finais (garantindo que a legenda apareça)
fig.update_layout(
    title=dict(x=0.5, xanchor="center"),
    xaxis_title="Faturamento por Cliente (R$)",
    yaxis_title="Quantidade de Clientes",
    bargap=0.1,
    showlegend=True  # Ativa a exibição da legenda no gráfico
)

fig.show()

In [46]:
# 1. Quantidade de clientes com faturamento acima da média
above_average_customers_count = (sales_by_client_data["revenue"] > mean_revenue).sum()

# 2. Faturamento total vindo dos clientes acima da média
above_average_total_revenue = sales_by_client_data.loc[
    sales_by_client_data["revenue"] > mean_revenue, "revenue"
].sum()

# 3. Quantidade de clientes com faturamento percentual acima da media e o faturamento percentual desses clientes em relação ao total
above_average_customers_percentage = above_average_customers_count / total_customers * 100
above_average_revenue_percentage = above_average_total_revenue / revenue * 100

In [47]:
kpi_card(
    title="Quantidade de clientes com faturamento acima da média.",
    value=format_number(above_average_customers_count),
    description=f"{format_number(above_average_customers_percentage)}% dos clientes"
)

In [48]:
kpi_card(
    title="Faturamento gerado pelos clientes com valores acima da média.",
    value=format_brl(above_average_total_revenue),
    description=f"{format_number(above_average_revenue_percentage)}% da receita"
)


**Resposta:** A receita não depende de um único cliente; o maior faturamento individual é de R$ 13.664,08. Ainda assim, há concentração em uma parcela da base de consumidores: 29,28% dos clientes representam 63,9% da receita, equivalente a R$ 9.810.944,74.

**Recomendação:**

Embora a receita não esteja concentrada em um único cliente, uma parcela de aproximadamente 29,28% da base é responsável por 63,9% do faturamento. Isso evidencia a importância desse grupo para os resultados da empresa. Recomenda-se desenvolver estratégias de retenção e fidelização, com programas de fidelidade, benefícios exclusivos e ações de relacionamento, para reduzir o risco de perda de receita. Também se recomenda implementar campanhas de upsell e cross-sell para aumentar o ticket médio dos demais clientes e estimular uma distribuição mais equilibrada do faturamento.

## Conclusão e Limitações da Análise

**Conclusão:** Com base na definição de saúde adotada, nas métricas estabelecidas e nos resultados obtidos, conclui-se que o crescimento da empresa é saudável. Há oportunidades para fidelizar os novos clientes adquiridos, além de um grupo de clientes que representa uma parcela significativa da receita.

**Limitações:** Para realizar uma análise mais completa da sustentabilidade do crescimento, seria necessário avaliar a retenção de clientes, a recorrência da receita, a evolução das margens e a concentração das vendas por vendedor ou região. Essas análises, no entanto, não foram contempladas neste trabalho, que se limita a apresentar um diagnóstico geral dos mecanismos de crescimento da receita. Além disso, os dados de 2018 estão incompletos devido ao truncamento da série histórica ao final do período. Portanto, as comparações devem considerar esse recorte, uma vez que os resultados não representam um ano completo.